# Fine-tune CXR-CLIP trên Colab (V1.1, B14–B15)

Hướng dẫn đầy đủ: `training/colab/README.md`. Trước khi chạy: **Runtime → Change runtime type → A100** (hoặc L4; bản miễn phí: **T4**, để tab mở và laptop không ngủ).

- Chạy lần lượt từng ô. Ô **kiểm tra** (bước 3, 4) mà báo FAIL thì **dừng lại**, đừng chạy tiếp.
- Mọi lượt train ghi vào `MyDrive/cxr_v11/runs/` và tự resume: mất kết nối thì chạy lại đúng ô đó.
- Runtime còn kết nối là còn tốn CU, kể cả khi ngồi không. Ô cuối tự trả GPU.

In [ ]:
# 1. Drive + GPU
# ruff: noqa: E501, F821, B007  (IPython shell lines, `_exit_code` is set by `!`)
from google.colab import drive

drive.mount('/content/drive')
BUNDLE = '/content/drive/MyDrive/cxr_v11'
WORK = '/content/work'
!nvidia-smi --query-gpu=name,memory.total --format=csv
!test -f {BUNDLE}/manifest.json && echo bundle found || echo 'BUNDLE MISSING: upload cxr_v11 to MyDrive'

In [ ]:
# 2. Code + the laptop's exact environment: a Python 3.12 venv synced from training/uv.lock.
#    Colab's own Python is newer (no tokenizers 0.19.1 / torch 2.4.1 wheels) and ships other
#    torch / transformers versions, so it is never used (WORKLOG T32). About 2-4 min.
!mkdir -p {WORK} && unzip -q -o {BUNDLE}/code.zip -d {WORK}
%cd {WORK}
!pip install -q uv
!uv venv -q --clear --python 3.12 /content/venv
!uv pip sync -q --python /content/venv/bin/python training/colab/pylock.colab.toml
assert _exit_code == 0, 'INSTALL FAIL: send the output above'
PY = '/content/venv/bin/python'
!{PY} -c "import sys, torch, transformers, peft; print(sys.version.split()[0], torch.__version__, transformers.__version__, peft.__version__, torch.cuda.get_device_name(0), 'native bf16', torch.cuda.is_bf16_supported(including_emulation=False))"
assert _exit_code == 0, 'NO GPU: Runtime > Change runtime type > T4 / L4 / A100'

In [ ]:
# 3. Data on the local disk, every file checked against manifest.json (1-3 min)
%cd {WORK}
!{PY} training/colab/colab_run.py setup --bundle {BUNDLE}
assert _exit_code == 0, 'SETUP FAIL: a file differs from the laptop, upload the bundle again'

In [ ]:
# 4. This GPU must reproduce the laptop's zero-shot pool-val P@5 (about 1 min)
!{PY} training/colab/colab_run.py check-env --bundle {BUNDLE}
assert _exit_code == 0, 'ENV FAIL: do not train, send the output above'

In [ ]:
# 5. Gate: a short FT6 run (20k images x 2 epochs) must not collapse or fall clearly below zero-shot
CU_PER_HOUR = 0  # from Colab's Resources panel, e.g. 5.4; 0 = do not project the cost
!{PY} training/colab/colab_run.py train --bundle {BUNDLE} --variant FT6 --lr 3e-5 --subset 20000 --epochs 2 --tag gate
!{PY} training/colab/colab_run.py gate --bundle {BUNDLE} --run gate_FT6_lr3e-05 --cu-per-hour {CU_PER_HOUR}
assert _exit_code == 0, 'GATE FAIL: do not start the full runs, send the output above'

In [ ]:
# 6. B14: lr sweep of FT6 (D33 grid), full data, 10 epochs with early stopping
for lr in ['3e-5', '1e-4', '1e-5']:
    !{PY} training/colab/colab_run.py train --bundle {BUNDLE} --variant FT6 --lr {lr}
!{PY} training/colab/colab_run.py summary --bundle {BUNDLE}

In [ ]:
# 7. The paper's recipe (FT1) at the lr FT6 chose: the reference row of the results table
BEST_LR = '3e-5'  # copy 'FT6: best lr ...' from the summary above
!{PY} training/colab/colab_run.py train --bundle {BUNDLE} --variant FT1 --lr {BEST_LR}
!{PY} training/colab/colab_run.py summary --bundle {BUNDLE}

In [ ]:
# 8. Release the GPU so no more CU is spent (the runs stay on Drive)
from google.colab import runtime

runtime.unassign()